## Evaluate Presidio Analyzer using the Presidio Evaluator framework

In [1]:
# install presidio evaluator via pip if not yet installed

#!pip install presidio-evaluator

In [2]:
from pathlib import Path
from pprint import pprint
from collections import Counter
from typing import Dict, List
import json

from presidio_evaluator import InputSample
from presidio_evaluator.evaluation import SpanEvaluator, ModelError, Plotter
from presidio_evaluator.experiment_tracking import get_experiment_tracker

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

%reload_ext autoreload
%autoreload 2

/Users/jyu36/code/ic-llm/presidio/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/Users/jyu36/code/ic-llm/presidio/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


stanza and spacy_stanza are not installed
Flair is not installed by default


In [3]:
dataset_name = "data_person_10_zh_presidio_format.json"
dataset = InputSample.read_dataset_json(Path(Path.cwd(), "data", dataset_name), token_model_version="zh_core_web_sm")
print(len(dataset))
print(dataset[0].full_text if dataset else "dataset is empty")

tokenizing input:   0%|          | 0/1 [00:00<?, ?it/s]

loading model zh_core_web_sm


tokenizing input: 100%|██████████| 1/1 [00:00<00:00,  1.29it/s]

1
陶立轩是一位62岁的男性软件开发工程师。他居住在浙江省杭州市西湖区文三路826号，持有身份证号330106196012139416。他的电子邮箱是taolixuan@163.com，联系电话是13857123456。他被诊断为肥胖症，症状包括体重超标、呼吸短促和关节不适。他的主治医生张敏华为他开具了布洛芬缓释胶囊。陶立轩的信用评分为608分，年收入为785,798.06元。最近有一笔来自印度法证服务的转账记录。


This dataset was auto generated. See more info here [Synthetic data generation](1_Generate_data.ipynb).

In [4]:
def get_entity_counts(dataset: List[InputSample]) -> Dict:
    """Return a dictionary with counter per entity type."""
    entity_counter = Counter()
    for sample in dataset:
        print("SAMPLE:", sample)
        print("NUM_TAGS:", len(sample.tags))
        for tag in sample.tags:
            print("TAG:", tag)
            entity_counter[tag] += 1
    return entity_counter

In [5]:
entity_counts = get_entity_counts(dataset)
print("Count per entity:")
pprint(entity_counts.most_common(), compact=True)

print("\nMin and max number of tokens in dataset: "\
f"Min: {min([len(sample.tokens) for sample in dataset])}, "\
f"Max: {max([len(sample.tokens) for sample in dataset])}")

print(f"Min and max sentence length in dataset: " \
f"Min: {min([len(sample.full_text) for sample in dataset])}, "\
f"Max: {max([len(sample.full_text) for sample in dataset])}")

print("\nExample InputSample:")
print(dataset[0])

SAMPLE: Full text: 陶立轩是一位62岁的男性软件开发工程师。他居住在浙江省杭州市西湖区文三路826号，持有身份证号330106196012139416。他的电子邮箱是taolixuan@163.com，联系电话是13857123456。他被诊断为肥胖症，症状包括体重超标、呼吸短促和关节不适。他的主治医生张敏华为他开具了布洛芬缓释胶囊。陶立轩的信用评分为608分，年收入为785,798.06元。最近有一笔来自印度法证服务的转账记录。
Spans: [Span(type: PERSON, value: 陶立轩, char_span: [0: 3]), Span(type: AGE, value: 62, char_span: [6: 8]), Span(type: STREET_ADDRESS, value: 浙江省杭州市西湖区文三路826号, char_span: [24: 40]), Span(type: EMAIL_ADDRESS, value: taolixuan@163.com, char_span: [73: 90]), Span(type: PHONE_NUMBER, value: 13857123456, char_span: [96: 107]), Span(type: PERSON, value: 张敏华, char_span: [142: 145]), Span(type: PERSON, value: 陶立轩, char_span: [158: 161])]

NUM_TAGS: 98
TAG: PERSON
TAG: PERSON
TAG: O
TAG: O
TAG: O
TAG: AGE
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: STREET_ADDRESS
TAG: STREET_ADDRESS
TAG: STREET_ADDRESS
TAG: STREET_ADDRESS
TAG: STREET_ADDRESS
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: EMAIL_ADDRESS
TAG: EM

In [6]:
print("A few examples sentences containing each entity:\n")
for entity in entity_counts.keys():
    samples = [sample for sample in dataset if entity in set(sample.tags)]
    if len(samples) > 1 and entity != "O":
        print(f"Entity: <{entity}> two example sentences:\n"
              f"\n1) {samples[0].full_text}"
              f"\n2) {samples[1].full_text}"
              f"\n------------------------------------\n")

A few examples sentences containing each entity:



In [7]:
from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider

# Loading the vanilla Analyzer Engine, with the default NER model.
configuration = {
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "zh", "model_name": "zh_core_web_sm"}],
}
provider = NlpEngineProvider(nlp_configuration=configuration)
nlp_engine_with_sm = provider.create_engine()
analyzer_engine = AnalyzerEngine(nlp_engine=nlp_engine_with_sm, supported_languages=["zh"], default_score_threshold=0.4)

In [8]:
from presidio_evaluator.models import  PresidioAnalyzerWrapper

entities_mapping=PresidioAnalyzerWrapper.presidio_entities_map # default mapping
pprint(entities_mapping)

dataset = SpanEvaluator.align_entity_types(
    dataset, 
    entities_mapping=entities_mapping, 
    allow_missing_mappings=True
)
new_entity_counts = get_entity_counts(dataset)
print("\nCount per entity after alignment:")
pprint(new_entity_counts.most_common(), compact=True)

dataset_entities = list(new_entity_counts.values())


{'ADDRESS': 'LOCATION',
 'AGE': 'AGE',
 'BIRTHDAY': 'DATE_TIME',
 'CITY': 'LOCATION',
 'CREDIT_CARD': 'CREDIT_CARD',
 'CREDIT_CARD_NUMBER': 'CREDIT_CARD',
 'DATE': 'DATE_TIME',
 'DATE_OF_BIRTH': 'DATE_TIME',
 'DATE_TIME': 'DATE_TIME',
 'DOB': 'DATE_TIME',
 'DOMAIN': 'URL',
 'DOMAIN_NAME': 'URL',
 'EMAIL': 'EMAIL_ADDRESS',
 'EMAIL_ADDRESS': 'EMAIL_ADDRESS',
 'FACILITY': 'LOCATION',
 'FIRST_NAME': 'PERSON',
 'GPE': 'LOCATION',
 'HCW': 'PERSON',
 'HOSP': 'ORGANIZATION',
 'HOSPITAL': 'ORGANIZATION',
 'IBAN': 'IBAN_CODE',
 'IBAN_CODE': 'IBAN_CODE',
 'ID': 'ID',
 'IP_ADDRESS': 'IP_ADDRESS',
 'LAST_NAME': 'PERSON',
 'LOC': 'LOCATION',
 'LOCATION': 'LOCATION',
 'NAME': 'PERSON',
 'NATIONALITY': 'NRP',
 'NORP': 'NRP',
 'NRP': 'NRP',
 'O': 'O',
 'ORG': 'ORGANIZATION',
 'ORGANIZATION': 'ORGANIZATION',
 'PATIENT': 'PERSON',
 'PATORG': 'ORGANIZATION',
 'PER': 'PERSON',
 'PERSON': 'PERSON',
 'PHONE': 'PHONE_NUMBER',
 'PHONE_NUMBER': 'PHONE_NUMBER',
 'PREFIX': 'TITLE',
 'SSN': 'US_SSN',
 'STAFF': 'PE

In [9]:
# Test Analyzer
text=dataset[0].full_text
res = analyzer_engine.analyze(text=text, 
                              language="zh", 
                              return_decision_process=True)
for result in res:
    print(f"\nEntity: {result.entity_type}, Text: {text[result.start:result.end]}\n\nAnalysis explanation:")
    pprint(result.analysis_explanation)


Entity: EMAIL_ADDRESS, Text: 他的电子邮箱是taolixuan@163.com

Analysis explanation:
{'recognizer': 'EmailRecognizer', 'pattern_name': 'Email (Medium)', 'pattern': "\\b((([!#$%&'*+\\-/=?^_`{|}~\\w])|([!#$%&'*+\\-/=?^_`{|}~\\w][!#$%&'*+\\-/=?^_`{|}~\\.\\w]{0,}[!#$%&'*+\\-/=?^_`{|}~\\w]))[@]\\w+([-.]\\w+)*\\.\\w+([-.]\\w+)*)\\b", 'original_score': 0.5, 'score': 1.0, 'textual_explanation': 'Detected by `EmailRecognizer` using pattern `Email (Medium)`', 'score_context_improvement': 0, 'supportive_context_word': '', 'validation_result': True, 'regex_flags': 26}

Entity: DATE_TIME, Text: 62岁

Analysis explanation:
{'recognizer': 'SpacyRecognizer', 'pattern_name': None, 'pattern': None, 'original_score': 0.85, 'score': 0.85, 'textual_explanation': "Identified as DATE_TIME by Spacy's Named Entity Recognition", 'score_context_improvement': 0, 'supportive_context_word': '', 'validation_result': None, 'regex_flags': None}

Entity: ORGANIZATION, Text: 浙江省杭州市西湖区文三路826号

Analysis explanation:
{'recognizer'

In [10]:
# 找到包含"51岁"的样本，打印其 token 与 tag 对应关系
for i, s in enumerate(dataset):
    if "51岁" in s.full_text:
        print(f"sample {i}: {s.full_text[:40]}")
        for tok, tag in zip(s.tokens, s.tags):
            if "51" in str(tok) or tag != "O":
                print(f"  tok={tok!r:15}  tag={tag}")
        print()

In [11]:
# Set up the experiment tracker to log the experiment for reproducibility
experiment = get_experiment_tracker()

# Create the evaluator object
evaluator = SpanEvaluator(model=analyzer_engine)


# Track model and dataset params
params = {"dataset_name": dataset_name, 
          "model_name": evaluator.model.name}
params.update(evaluator.model.to_log())
experiment.log_parameters(params)
experiment.log_dataset_hash(dataset)
experiment.log_parameter("entity_mappings", json.dumps(entities_mapping))

--------
Entities supported by this Presidio Analyzer instance:
EMAIL, NRP, PHONE_NUMBER, IBAN_CODE, AGE, DATE_TIME, URL, ORGANIZATION, LOCATION, ID, MAC_ADDRESS, EMAIL_ADDRESS, PERSON, MEDICAL_LICENSE, IP_ADDRESS, CRYPTO


/Users/jyu36/code/ic-llm/presidio/.venv/lib/python3.12/site-packages/presidio_evaluator/evaluation/base_evaluator.py:83: UserWarning: skip words not provided, using default skip words. If you want the evaluation to not use skip words, pass skip_words=[]
  warnings.warn("skip words not provided, using default skip words. "


## 6. Run experiment

In [12]:
%%time

## Run experiment

evaluation_results = evaluator.evaluate_all(dataset)
results = evaluator.calculate_score(evaluation_results)

# Track experiment results
experiment.log_metrics(results.to_log())
entities, confmatrix = results.to_confusion_matrix()
experiment.log_confusion_matrix(matrix=confmatrix, 
                                labels=entities)

# end experiment
experiment.end()

Running model PresidioAnalyzerWrapper on dataset...
Finished running model on dataset
saving experiment data to /Users/jyu36/code/ic-llm/presidio/experiment_20260319-194800.json
CPU times: user 27 ms, sys: 2.29 ms, total: 29.3 ms
Wall time: 31.9 ms


In [13]:
pprint({"PII F":results.pii_f, "PII recall": results.pii_recall, "PII precision": results.pii_precision})

{'PII F': 0.42857142857142855,
 'PII precision': 0.42857142857142855,
 'PII recall': 0.42857142857142855}


### 7a. False positives
#### Most common false positive tokens:

In [14]:
results.model_errors

[<ModelError type: ErrorType.FN, Annotation = O, prediction = PERSON, Token = 陶立 轩, Full text = 陶立 轩, Metadata = None,
 <ModelError type: ErrorType.FN, Annotation = O, prediction = AGE, Token = 62, Full text = 62, Metadata = None,
 <ModelError type: ErrorType.FP, Annotation = DATE_TIME, prediction = O, Token = 62 岁, Full text = 62 岁, Metadata = None,
 <ModelError type: ErrorType.WrongEntity, Annotation = ORGANIZATION, prediction = LOCATION, Token = 浙江省 杭州市 西湖区 文三路 826号, Full text = 浙江省 杭州市 西湖区 文三路 826号, Metadata = None,
 <ModelError type: ErrorType.FN, Annotation = O, prediction = LOCATION, Token = 浙江省 杭州市 西湖区 文三路 826号, Full text = 浙江省 杭州市 西湖区 文三路 826号, Metadata = None,
 <ModelError type: ErrorType.FP, Annotation = ORGANIZATION, prediction = O, Token = 浙江省 杭州市 西湖区 文三路 826号, Full text = 浙江省 杭州市 西湖区 文三路 826号, Metadata = None,
 <ModelError type: ErrorType.FN, Annotation = O, prediction = EMAIL_ADDRESS, Token = taolixuan 163.com, Full text = taolixuan @ 163.com, Metadata = None,
 <ModelErr

In [15]:
ModelError.most_common_fp_tokens(results.model_errors)

Most common false positive tokens:
[('浙江省 杭州市 西湖区 文三路 826号', 2),
 ('布洛芬', 2),
 ('印度', 2),
 ('62 岁', 1),
 ('他 的 电子 邮箱 是 taolixuan 163.com', 1)]
---------------
Example sentence with each FP token:
	- 浙江省 杭州市 西湖区 文三路 826号 (`浙江省 杭州市 西湖区 文三路 826号` pred as O)
	- 布洛芬 (`布洛芬` pred as PERSON)
	- 印度 (`印度` pred as LOCATION)
	- 62 岁 (`62 岁` pred as O)
	- 他 的 电子 邮箱 是 taolixuan @ 163.com (`他 的 电子 邮箱 是 taolixuan 163.com` pred as O)


[('浙江省 杭州市 西湖区 文三路 826号', 2),
 ('布洛芬', 2),
 ('印度', 2),
 ('62 岁', 1),
 ('他 的 电子 邮箱 是 taolixuan 163.com', 1)]

#### More FP analysis

In [16]:
fps_df = ModelError.get_fps_dataframe(results.model_errors)
fps_df[["full_text", "token", "annotation", "prediction"]].head(20)

,full_text,token,annotation,prediction
0,62 岁,62 岁,DATE_TIME,O
1,浙江省 杭州市 西湖区 文三路 826号,浙江省 杭州市 西湖区 文三路 826号,ORGANIZATION,O
2,他 的 电子 邮箱 是 taolixuan @ 163.com,他 的 电子 邮箱 是 taolixuan 163.com,EMAIL_ADDRESS,O
3,布洛芬,布洛芬,O,PERSON
4,印度,印度,O,LOCATION
5,布洛芬,布洛芬,O,PII
6,印度,印度,O,PII
7,浙江省 杭州市 西湖区 文三路 826号,浙江省 杭州市 西湖区 文三路 826号,ORGANIZATION,LOCATION
8,浙江省 杭州市 西湖区 文三路 826号,浙江省 杭州市 西湖区 文三路 826号,ORGANIZATION,LOCATION


### 7b. False negatives (FN)

#### Most common false negative examples + a few samples with FN

In [17]:
ModelError.most_common_fn_tokens(results.model_errors, n=15)

Most common false negative tokens:
[('陶立 轩', 2), ('浙江省 杭州市 西湖区 文三路 826号', 2), ('62', 1), ('taolixuan 163.com', 1)]
---------------
Example sentence with each FN token:
	- 陶立 轩 (`陶立 轩` annotated as O)
	- 浙江省 杭州市 西湖区 文三路 826号 (`浙江省 杭州市 西湖区 文三路 826号` annotated as O)
	- 62 (`62` annotated as O)
	- taolixuan @ 163.com (`taolixuan 163.com` annotated as O)


[('陶立 轩', 2), ('浙江省 杭州市 西湖区 文三路 826号', 2), ('62', 1), ('taolixuan 163.com', 1)]

#### More FN analysis

In [18]:
fns_df = ModelError.get_fns_dataframe(results.model_errors)

In [19]:
fns_df[["full_text", "token", "annotation", "prediction"]].head(20)

,full_text,token,annotation,prediction
0,陶立 轩,陶立 轩,O,PERSON
1,62,62,O,AGE
2,浙江省 杭州市 西湖区 文三路 826号,浙江省 杭州市 西湖区 文三路 826号,O,LOCATION
3,taolixuan @ 163.com,taolixuan 163.com,O,EMAIL_ADDRESS
4,陶立 轩,陶立 轩,O,PERSON
5,浙江省 杭州市 西湖区 文三路 826号,浙江省 杭州市 西湖区 文三路 826号,ORGANIZATION,LOCATION
6,浙江省 杭州市 西湖区 文三路 826号,浙江省 杭州市 西湖区 文三路 826号,ORGANIZATION,LOCATION


In [20]:
# 事件级视图（补加版）：一个 token 事件只保留一条记录
# 将 WrongEntity 及其派生的 O->X / Y->O 记录合并

import pandas as pd


def _err_to_row(e, idx):
    return {
        "idx": idx,
        "error_type": str(getattr(e, "error_type", None)),
        "annotation": str(getattr(e, "annotation", None)),
        "prediction": str(getattr(e, "prediction", None)),
        "token": getattr(e, "token", None),
        "full_text": getattr(e, "full_text", None),
        "metadata": getattr(e, "metadata", None),
        "explanation": getattr(e, "explanation", None),
        "_used": False,
    }


def collapse_model_errors_to_events(model_errors, token_filter=None):
    rows = [_err_to_row(e, i) for i, e in enumerate(model_errors)]
    if token_filter is not None:
        rows = [r for r in rows if r["token"] == token_filter]

    wrong = [r for r in rows if r["error_type"].endswith("WrongEntity")]
    fn_like = [
        r
        for r in rows
        if r["error_type"].endswith("FN") and r["annotation"] == "O" and r["prediction"] != "O"
    ]
    fp_like = [
        r
        for r in rows
        if r["error_type"].endswith("FP") and r["annotation"] != "O" and r["prediction"] == "O"
    ]

    def same_context(a, b):
        return (
            a["token"] == b["token"]
            and a["full_text"] == b["full_text"]
            and str(a["metadata"]) == str(b["metadata"])
        )

    collapsed = []

    for w in wrong:
        if w["_used"]:
            continue

        ann = w["annotation"]
        pred = w["prediction"]

        match_fn = None
        for f in fn_like:
            if not f["_used"] and same_context(w, f) and f["prediction"] == pred:
                match_fn = f
                break

        match_fp = None
        for f in fp_like:
            if not f["_used"] and same_context(w, f) and f["annotation"] == ann:
                match_fp = f
                break

        w["_used"] = True
        if match_fn:
            match_fn["_used"] = True
        if match_fp:
            match_fp["_used"] = True

        collapsed.append(
            {
                "event_type": "WrongEntity",
                "annotation": ann,
                "prediction": pred,
                "token": w["token"],
                "explanation": w["explanation"],
                "source_idxs": [x for x in [w["idx"], match_fn["idx"] if match_fn else None, match_fp["idx"] if match_fp else None] if x is not None],
            }
        )

    for r in rows:
        if r["_used"]:
            continue
        collapsed.append(
            {
                "event_type": r["error_type"].split(".")[-1],
                "annotation": r["annotation"],
                "prediction": r["prediction"],
                "token": r["token"],
                "explanation": r["explanation"],
                "source_idxs": [r["idx"]],
            }
        )

    return pd.DataFrame(collapsed)


# 全量事件级视图
events_df = collapse_model_errors_to_events(results.model_errors)
print(f"raw model_errors: {len(results.model_errors)}")
print(f"collapsed events: {len(events_df)}")
display(events_df.head(50))

raw model_errors: 13
collapsed events: 11


,event_type,annotation,prediction,token,explanation,source_idxs
0,WrongEntity,ORGANIZATION,LOCATION,浙江省 杭州市 西湖区 文三路 826号,"Wrong entity type: LOCATION detected as ORGANIZATION, iou=1.00 compared to threshold=0.9","[3, 4, 5]"
1,FN,O,PERSON,陶立 轩,Entity PERSON not detected.,[0]
2,FN,O,AGE,62,Entity AGE not detected. iou with DATE_TIME=0.67 compared to threshold=0.9,[1]
3,FP,DATE_TIME,O,62 岁,"Entity DATE_TIME falsely detected, iou=0.67 compared to threshold=0.9",[2]
4,FN,O,EMAIL_ADDRESS,taolixuan 163.com,Entity EMAIL_ADDRESS not detected due to low iou=0.71 compared to threshold=0.9,[6]
5,FP,EMAIL_ADDRESS,O,他 的 电子 邮箱 是 taolixuan 163.com,"Entity EMAIL_ADDRESS falsely detected, iou=0.71 compared to threshold=0.9",[7]
6,FN,O,PERSON,陶立 轩,Entity PERSON not detected.,[8]
7,FP,O,PERSON,布洛芬,False prediction with no overlap: PERSON,[9]
8,FP,O,LOCATION,印度,False prediction with no overlap: LOCATION,[10]
9,FP,O,PII,布洛芬,False prediction with no overlap: PII,[11]


In [21]:
display(events_df[events_df['token'] == '布洛芬'].head(50))

,event_type,annotation,prediction,token,explanation,source_idxs
7,FP,O,PERSON,布洛芬,False prediction with no overlap: PERSON,[9]
9,FP,O,PII,布洛芬,False prediction with no overlap: PII,[11]
